In [ ]:
"""
=============================================================================
NOTEBOOK — Strong Lightweight Contrastive Learning (Complete Version)
=============================================================================
"""

import os, random, time, logging
from collections import defaultdict
from typing import Dict, List
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report,
                             roc_curve, auc, confusion_matrix)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import json

sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 14, 'axes.labelsize': 14, 'axes.titlesize': 16})

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/research/contrastive learning/compare_complete/"
os.makedirs(DRIVE_PATH, exist_ok=True)
log.info(f"Results will be saved to: {DRIVE_PATH}")

# ========================== CFG ==========================

CFG = {
    "model_name"        : "microsoft/codebert-base",
    "max_length"        : 512,
    "train_sample_size" : 32_000,
    "val_size"          : 8_000,
    "test_size"         : 12_000,
    "epochs"            : 5,
    "batch_size"        : 16,
    "grad_accum_steps"  : 4,
    "eval_batch_size"   : 64,
    "lr"                : 2.0e-5,
    "weight_decay"      : 0.04,
    "warmup_ratio"      : 0.18,
    "gradient_clip"     : 1.0,

    "temperature"       : 0.09,
    "hard_neg_weight"   : 1.45,
    "proj_dim"          : 448,

    "unfreeze_layers"   : 5,
    "use_mean_pooling"  : True,

    "map_k"             : 500,
    "map_n_queries"     : 12000,
    "device"            : "cuda",
    "save_dir"          : "./checkpoints_strong_fixed",
    "fp16"              : True,
    "dpi"               : 600,
}
os.makedirs(CFG["save_dir"], exist_ok=True)

# =============================================================================
# 1. Data Loading
# =============================================================================
def load_poj104_with_val(train_size: int = 32_000,
                         val_size:   int =  8_000,
                         test_size:  int = 12_000):
    log.info("Loading POJ-104...")
    ds       = load_dataset("semeru/Code-Code-CloneDetection-POJ104")
    train_raw = ds["train"]
    val_raw   = ds["validation"]
    test_raw  = ds["test"]

    if len(train_raw) > train_size:
        labels = [int(lbl) for lbl in train_raw["label"]]
        idx, _ = train_test_split(range(len(train_raw)), train_size=train_size,
                                  random_state=SEED, stratify=labels)
        train_raw = train_raw.select(idx)

    if len(val_raw) > val_size:
        labels = [int(lbl) for lbl in val_raw["label"]]
        idx, _ = train_test_split(range(len(val_raw)), train_size=val_size,
                                  random_state=SEED, stratify=labels)
        val_raw = val_raw.select(idx)

    if len(test_raw) > test_size:
        labels = [int(lbl) for lbl in test_raw["label"]]
        idx, _ = train_test_split(range(len(test_raw)), train_size=test_size,
                                  random_state=SEED, stratify=labels)
        test_raw = test_raw.select(idx)

    log.info(f"Final splits → Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")
    return train_raw, val_raw, test_raw


def group_by_label(dataset) -> Dict[int, List[int]]:
    groups = defaultdict(list)
    for i, lbl in enumerate(dataset["label"]):
        groups[int(lbl)].append(i)
    return dict(groups)

# =============================================================================
# 2. Datasets
# =============================================================================
class InBatchDataset(Dataset):
    def __init__(self, raw_ds, groups, tokenizer, max_len):
        self.raw   = raw_ds
        self.groups = groups
        self.tok   = tokenizer
        self.max_len = max_len
        self.items = [
            (idx, pid)
            for pid, idxs in groups.items()
            if len(idxs) >= 2
            for idx in idxs
        ]
        log.info(f"InBatchDataset: {len(self.items)} usable snippets")

    def _encode(self, code):
        enc = self.tok(code, max_length=self.max_len, padding="max_length",
                       truncation=True, return_tensors="pt")
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        snippet_idx, pid = self.items[idx]
        ids, mask = self._encode(self.raw[snippet_idx]["code"])
        return ids, mask, torch.tensor(pid, dtype=torch.long)


class EmbeddingDataset(Dataset):
    def __init__(self, raw_ds, tokenizer, max_len):
        self.raw    = raw_ds
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.raw)

    def __getitem__(self, idx):
        enc = self.tok(self.raw[idx]["code"], max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return (enc["input_ids"].squeeze(0),
                enc["attention_mask"].squeeze(0),
                torch.tensor(int(self.raw[idx]["label"]), dtype=torch.long))

# =============================================================================
# 3. Model
# =============================================================================
class StrongContrastiveModel(nn.Module):
    def __init__(self, model_name, proj_dim=448, unfreeze_layers=5):
        super().__init__()
        self.bert = RobertaModel.from_pretrained(model_name)

        for param in self.bert.parameters():
            param.requires_grad = False

        for i, layer in enumerate(self.bert.encoder.layer):
            if i >= (12 - unfreeze_layers):
                for param in layer.parameters():
                    param.requires_grad = True

        log.info(f"Last {unfreeze_layers} encoder layers unfrozen.")

        hidden = self.bert.config.hidden_size

        self.projector = nn.Sequential(
            nn.Linear(hidden, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim),
        )

    def encode(self, ids, mask):
        h = self.bert(ids, mask).last_hidden_state.mean(dim=1)
        p = self.projector(h)
        return F.normalize(p, dim=-1)

    def forward(self, ids, mask):
        return self.encode(ids, mask)

# =============================================================================
# 4. Loss
# =============================================================================
class MultiPositiveContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.09, hard_neg_weight=1.45):
        super().__init__()
        self.tau   = temperature
        self.alpha = hard_neg_weight

    def forward(self, z, labels):
        N = z.size(0)
        sim = torch.mm(z, z.T) / self.tau

        labels   = labels.view(-1, 1)
        pos_mask = labels.eq(labels.T).float()
        eye      = torch.eye(N, device=z.device)
        pos_mask = pos_mask - eye
        neg_mask = 1.0 - labels.eq(labels.T).float()

        with torch.no_grad():
            sim_det  = sim.detach()
            neg_sims = sim_det * neg_mask
            mean_sim = (neg_sims.sum(dim=1, keepdim=True) / (neg_mask.sum(dim=1, keepdim=True) + 1e-8))
            hard_mask  = (neg_sims > mean_sim).float() * neg_mask
            weight_mat = torch.ones_like(sim_det) + (self.alpha - 1.0) * hard_mask

        sim_max  = sim.detach().max(dim=1, keepdim=True).values
        exp_sim  = torch.exp(sim - sim_max)
        denom    = (exp_sim * weight_mat * (1 - eye)).sum(dim=1)
        log_prob = (sim - sim_max) - torch.log(denom.unsqueeze(1) + 1e-8)

        n_pos = pos_mask.sum(dim=1).clamp(min=1)
        loss  = -(log_prob * pos_mask).sum(dim=1) / n_pos

        valid = pos_mask.sum(dim=1) > 0
        return loss[valid].mean() if valid.any() else loss.mean()

# =============================================================================
# 5. Training Utilities
# =============================================================================
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self):    self.sum = self.count = 0.0
    def update(self, v, n=1): self.sum += v * n; self.count += n
    @property
    def avg(self): return self.sum / max(self.count, 1e-8)


@torch.no_grad()
def validate(model, val_loader, criterion, device):
    model.eval()
    meter = AverageMeter()
    for batch in tqdm(val_loader, desc="Validating", leave=False):
        ids, mask, labels = [b.to(device) for b in batch]
        with torch.amp.autocast('cuda', enabled=True):
            z    = model(ids, mask)
            loss = criterion(z, labels)
        meter.update(loss.item(), ids.size(0))
    model.train()
    return meter.avg


def save_loss_curve(train_losses, val_losses, save_dir, dpi=600):
    plt.figure(figsize=(9, 6))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, 'b-', linewidth=2.5, label='Train Loss')
    plt.plot(epochs, val_losses,   'r-', linewidth=2.5, label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=13, loc='upper right', frameon=True, shadow=True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'strong_fixed_loss_curve.pdf'),
                dpi=dpi, format='pdf', bbox_inches='tight')
    plt.close()

# =============================================================================
# 6. Training Loop
# =============================================================================
def train(model, train_loader, val_loader, cfg):
    device = cfg["device"]
    model  = model.to(device)

    optimizer    = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total_steps  = len(train_loader) * cfg["epochs"]
    warmup_steps = int(total_steps * cfg["warmup_ratio"])
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    criterion = MultiPositiveContrastiveLoss(
        temperature=cfg["temperature"],
        hard_neg_weight=cfg["hard_neg_weight"]
    ).to(device)

    scaler       = torch.amp.GradScaler('cuda', enabled=cfg["fp16"])
    train_losses = []
    val_losses   = []

    for epoch in range(cfg["epochs"]):
        model.train()
        train_meter = AverageMeter()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg['epochs']}")
        optimizer.zero_grad()

        for step, batch in enumerate(pbar):
            ids, mask, labels = [b.to(device) for b in batch]

            with torch.amp.autocast('cuda', enabled=cfg["fp16"]):
                z    = model(ids, mask)
                loss = criterion(z, labels)
                loss = loss / cfg["grad_accum_steps"]

            scaler.scale(loss).backward()

            if (step + 1) % cfg["grad_accum_steps"] == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            train_meter.update(loss.item() * cfg["grad_accum_steps"], ids.size(0))
            pbar.set_postfix(train_loss=f"{train_meter.avg:.4f}")

        train_loss = train_meter.avg
        train_losses.append(train_loss)

        val_loss = validate(model, val_loader, criterion, device)
        val_losses.append(val_loss)

        log.info(f"Epoch {epoch+1:2d} → Train: {train_loss:.4f} | Val: {val_loss:.4f}")
        print(f"📊 Epoch {epoch+1:2d} → Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    save_loss_curve(train_losses, val_losses, DRIVE_PATH, cfg["dpi"])
    torch.save(model.state_dict(), os.path.join(cfg["save_dir"], "best_model.bin"))
    return model

# =============================================================================
# 7. Evaluation
# =============================================================================
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    embs, lbls = [], []
    for ids, mask, lbl in tqdm(loader, desc="Extracting embeddings"):
        ids, mask = ids.to(device), mask.to(device)
        h = model.encode(ids, mask)
        embs.append(h.cpu().numpy())
        lbls.extend(lbl.numpy().tolist())
    return np.vstack(embs), np.array(lbls, dtype=np.int32)


def compute_map_at_k(embs, labels, k=500, n_queries=12000, seed=SEED):
    rng    = np.random.RandomState(seed)
    n      = len(embs)
    norms  = np.linalg.norm(embs, axis=1, keepdims=True)
    embs_n = embs / (norms + 1e-8)
    q_idx  = rng.choice(n, size=min(n_queries, n), replace=False)
    aps    = []

    for qi in tqdm(q_idx, desc=f"MAP@{k}", leave=False):
        q_lbl = labels[qi]
        sims  = embs_n @ embs_n[qi]
        sims[qi] = -np.inf
        ranked = np.argsort(-sims)[:k]
        n_rel  = int((labels == q_lbl).sum()) - 1
        if n_rel <= 0: continue
        hits, prec_at_hit = 0, []
        for rank, ri in enumerate(ranked, 1):
            if labels[ri] == q_lbl:
                hits += 1
                prec_at_hit.append(hits / rank)
        ap = sum(prec_at_hit) / min(n_rel, k) if prec_at_hit else 0.0
        aps.append(ap)

    return float(np.mean(aps)) if aps else 0.0


def tune_threshold_on_val(val_embs, val_labels, val_groups, n_pairs=5000):
    log.info("Tuning threshold on validation set...")
    val_pids = list(val_groups.keys())
    pairs    = []
    half     = n_pairs // 2
    random.seed(SEED)

    while len(pairs) < half:
        pid  = random.choice(val_pids)
        idxs = val_groups[pid]
        if len(idxs) < 2: continue
        a, b = random.sample(idxs, 2)
        pairs.append((a, b, 1))

    while len(pairs) < n_pairs:
        p1, p2 = random.sample(val_pids, 2)
        a = random.choice(val_groups[p1])
        b = random.choice(val_groups[p2])
        pairs.append((a, b, 0))

    norms  = np.linalg.norm(val_embs, axis=1, keepdims=True)
    embs_n = val_embs / (norms + 1e-8)
    sims   = np.array([float(np.dot(embs_n[a], embs_n[b])) for a, b, _ in pairs])
    gts    = np.array([lbl for _, _, lbl in pairs])

    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(sims.min(), sims.max(), 200):
        preds = (sims >= t).astype(int)
        f1    = f1_score(gts, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)

    log.info(f"Best threshold: {best_t:.4f}  (val F1 = {best_f1:.4f})")
    return best_t


def generate_eval_pairs(test_raw, test_groups, n_pairs=10000, seed=SEED):
    random.seed(seed)
    pids  = list(test_groups.keys())
    pairs = []
    half  = n_pairs // 2

    while len(pairs) < half:
        pid  = random.choice(pids)
        idxs = test_groups[pid]
        if len(idxs) < 2: continue
        a, b = random.sample(idxs, 2)
        pairs.append((a, b, 1))

    while len(pairs) < n_pairs:
        p1, p2 = random.sample(pids, 2)
        a = random.choice(test_groups[p1])
        b = random.choice(test_groups[p2])
        pairs.append((a, b, 0))

    random.shuffle(pairs)
    return pairs


def save_confusion_matrix_pdf(cm, save_dir, dpi=600):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Clone', 'Clone'],
                yticklabels=['Non-Clone', 'Clone'])
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'strong_fixed_confusion_matrix.pdf'),
                dpi=dpi, format='pdf')
    plt.close()


def evaluate(model, test_raw, cfg, best_threshold):
    device = cfg["device"]
    tok    = RobertaTokenizer.from_pretrained(cfg["model_name"])

    start_pred = time.time()
    emb_ds = EmbeddingDataset(test_raw, tok, cfg["max_length"])
    emb_dl = DataLoader(emb_ds, batch_size=cfg["eval_batch_size"],
                        shuffle=False, num_workers=0, pin_memory=True)
    embs, labels = extract_embeddings(model, emb_dl, device)

    test_groups = group_by_label(test_raw)
    pairs       = generate_eval_pairs(test_raw, test_groups)

    norms  = np.linalg.norm(embs, axis=1, keepdims=True)
    embs_n = embs / (norms + 1e-8)
    sims   = np.array([float(np.dot(embs_n[a], embs_n[b])) for a, b, _ in pairs])
    gts    = np.array([lbl for _, _, lbl in pairs])

    pred_time  = time.time() - start_pred
    start_eval = time.time()

    preds = (sims >= best_threshold).astype(int)
    cm    = confusion_matrix(gts, preds)
    save_confusion_matrix_pdf(cm, DRIVE_PATH, cfg["dpi"])

    fpr, tpr, _ = roc_curve(gts, sims)
    roc_auc     = auc(fpr, tpr)

    roc_data = {"model": "Strong_Contrastive_Fixed", "fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": float(roc_auc)}
    with open(os.path.join(DRIVE_PATH, "strong_fixed_roc_data.json"), "w") as f:
        json.dump(roc_data, f, indent=2)

    f1   = f1_score(gts, preds, zero_division=0)
    acc  = accuracy_score(gts, preds)
    prec = precision_score(gts, preds, zero_division=0)
    rec  = recall_score(gts, preds, zero_division=0)

    class_report = classification_report(gts, preds, target_names=['Negative', 'Positive'], output_dict=True)
    map500 = compute_map_at_k(embs, labels, k=cfg["map_k"], n_queries=cfg["map_n_queries"])
    total_eval_time = time.time() - start_eval

    results = {
        "Method"               : "Strong Lightweight Contrastive (Fixed Projector)",
        "F1"                   : round(f1, 4),
        "Precision"            : round(prec, 4),
        "Recall"               : round(rec, 4),
        "Accuracy"             : round(acc, 4),
        "MAP@500"              : round(map500, 4),
        "AUC"                  : round(roc_auc, 4),
        "Prediction Time (s)"  : round(pred_time, 2),
        "Evaluation Time (s)"  : round(total_eval_time, 2),
        "Precision (Negative)" : round(class_report['Negative']['precision'], 4),
        "Recall (Negative)"    : round(class_report['Negative']['recall'], 4),
        "F1 (Negative)"        : round(class_report['Negative']['f1-score'], 4),
        "Precision (Positive)" : round(class_report['Positive']['precision'], 4),
        "Recall (Positive)"    : round(class_report['Positive']['recall'], 4),
        "F1 (Positive)"        : round(class_report['Positive']['f1-score'], 4),
    }
    return results, embs, labels

# =============================================================================
# 8. Main + Saving
# =============================================================================
def run(cfg=CFG):
    train_raw, val_raw, test_raw = load_poj104_with_val(
        cfg["train_sample_size"], cfg["val_size"], cfg["test_size"]
    )

    tok = RobertaTokenizer.from_pretrained(cfg["model_name"])

    train_groups = group_by_label(train_raw)
    train_ds = InBatchDataset(train_raw, train_groups, tok, cfg["max_length"])
    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)

    val_groups = group_by_label(val_raw)
    val_ds = InBatchDataset(val_raw, val_groups, tok, cfg["max_length"])
    val_dl = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=True,
                        num_workers=0, pin_memory=True, drop_last=True)

    model = StrongContrastiveModel(cfg["model_name"], cfg["proj_dim"], cfg["unfreeze_layers"])
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"Trainable parameters: {param_count/1e6:.3f}M")

    t_train = time.time()
    model = train(model, train_dl, val_dl, cfg)
    train_time = (time.time() - t_train) / 60

    val_emb_ds = EmbeddingDataset(val_raw, tok, cfg["max_length"])
    val_emb_dl = DataLoader(val_emb_ds, batch_size=cfg["eval_batch_size"], shuffle=False, num_workers=0, pin_memory=True)
    val_embs, val_labels = extract_embeddings(model, val_emb_dl, cfg["device"])
    best_threshold = tune_threshold_on_val(val_embs, val_labels, val_groups)

    results, embs, labels = evaluate(model, test_raw, cfg, best_threshold)
    results["Training Time (min)"] = round(train_time, 1)
    results["Parameters (M)"] = round(param_count / 1_000_000, 3)

    # ====================== SAVE EVERYTHING ======================
    log.info("Extracting train embeddings for saving...")
    train_emb_ds = EmbeddingDataset(train_raw, tok, cfg["max_length"])
    train_emb_dl = DataLoader(train_emb_ds, batch_size=cfg["eval_batch_size"], shuffle=False, num_workers=0, pin_memory=True)
    train_embs, train_labels_np = extract_embeddings(model, train_emb_dl, cfg["device"])

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    torch.save(model.state_dict(), f"{DRIVE_PATH}strong_model_{timestamp}.pt")
    np.save(f"{DRIVE_PATH}train_embs_{timestamp}.npy", train_embs)
    np.save(f"{DRIVE_PATH}val_embs_{timestamp}.npy", val_embs)
    np.save(f"{DRIVE_PATH}test_embs_{timestamp}.npy", embs)
    np.save(f"{DRIVE_PATH}train_labels_{timestamp}.npy", train_labels_np)
    np.save(f"{DRIVE_PATH}val_labels_{timestamp}.npy", val_labels)
    np.save(f"{DRIVE_PATH}test_labels_{timestamp}.npy", labels)

    with open(f"{DRIVE_PATH}results_{timestamp}.json", "w") as f:
        json.dump(results, f, indent=2)

    with open(f"{DRIVE_PATH}config_{timestamp}.json", "w") as f:
        json.dump(cfg, f, indent=2)

    eval_pairs = generate_eval_pairs(test_raw, group_by_label(test_raw))
    with open(f"{DRIVE_PATH}eval_pairs_{timestamp}.json", "w") as f:
        json.dump({"pairs": eval_pairs, "best_threshold": float(best_threshold)}, f, indent=2)

    # Summary
    summary = f"""STRONG CONTRASTIVE RESULTS - {time.strftime("%Y-%m-%d %H:%M:%S")}
F1: {results['F1']} | MAP@500: {results['MAP@500']} | AUC: {results['AUC']}
Training Time: {results['Training Time (min)']} min | Params: {results['Parameters (M)']}M
Files saved with timestamp: {timestamp}"""
    with open(f"{DRIVE_PATH}summary_{timestamp}.txt", "w") as f:
        f.write(summary)

    log.info(f"✅ All files successfully saved to: {DRIVE_PATH}")

    # Final Print
    print("\n" + "="*80)
    print("FINAL RESULTS — Strong Lightweight Contrastive")
    print("="*80)
    for key in ["F1", "Precision", "Recall", "Accuracy", "MAP@500", "AUC"]:
        print(f"{key:<25} {results[key]:>10.4f}")
    print("="*80)

    return results, model


if __name__ == "__main__":
    results, model = run(CFG)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/22.6M [00:00<?, ?B/s]

valid.jsonl: 0.00B [00:00, ?B/s]

...

Extracting embeddings:   0%|          | 0/125 [00:00<?, ?it/s]

Extracting embeddings:   0%|          | 0/188 [00:00<?, ?it/s]

MAP@500:   0%|          | 0/12000 [00:00<?, ?it/s]

Extracting embeddings:   0%|          | 0/500 [00:00<?, ?it/s]


FINAL RESULTS — Strong Lightweight Contrastive
F1                            0.9289
Precision                     0.9375
Recall                        0.9204
Accuracy                      0.9295
MAP@500                       0.7945
AUC                           0.9807
